In [13]:
pip install fasttext

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 kB 3.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached pybind11-2.13.6-py3-none-any.whl.metadata (9.5 kB)
Using cached pybind11-2.13.6-py3-none-any.whl (243 kB)
  Created wheel for fasttext: filename=fasttext-0.9.3-cp311-cp311-linux_x86_64.whl size=4313500 sha256=b0a1d53fc09cc89678982dfb51e2c96be36486142b66370460febeb8c58c13c3
  Stored in directory: /root/.cache/pip/wheels/65/4f/35/5057db0249224e9ab55a513fa6b79451473ceb7713017823c3
Successfully built fasttext


In [14]:
import re
import pandas as pd
import fasttext
from sklearn.model_selection import train_test_split

In [2]:
df= pd.read_csv("ecommerce_dataset.csv", names=["category", "description"], header=None)
print(df.shape)
df.head(3)

(50425, 2)


,category,description
0,Household,Paper Plane Design Framed Wall Hanging Motivat...
1,Household,"SAF 'Floral' Framed Painting (Wood, 30 inch x ..."
2,Household,SAF 'UV Textured Modern Art Print Framed' Pain...


In [3]:
df.dropna(inplace=True)
df.shape

(50424, 2)

In [4]:
df.category.unique()

array(['Household', 'Books', 'Clothing & Accessories', 'Electronics'],
      dtype=object)

In [5]:
df.category.replace("Clothing & Accessories", "Clothing_Accessories", inplace=True)

<ipython-input-5-5cbb1445e79b>:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df.category.replace("Clothing & Accessories", "Clothing_Accessories", inplace=True)


In [6]:
df['category'] = '__label__' + df['category'].astype(str)
df['category_description'] = df['category'] + ' ' + df['description']
df.head(3)

,category,description,category_description
0,__label__Household,Paper Plane Design Framed Wall Hanging Motivat...,__label__Household Paper Plane Design Framed W...
1,__label__Household,"SAF 'Floral' Framed Painting (Wood, 30 inch x ...",__label__Household SAF 'Floral' Framed Paintin...
2,__label__Household,SAF 'UV Textured Modern Art Print Framed' Pain...,__label__Household SAF 'UV Textured Modern Art...


# Pre-procesing

1.Remove punctuation
2.Remove extra space
3.Make the entire sentence lower case

In [8]:
def preprocess(text):
    text = re.sub(r'[^\w\s\']',' ', text)
    text = re.sub(' +', ' ', text)
    return text.strip().lower()

df['category_description'] = df['category_description'].map(preprocess)

# Train,Test split

In [11]:
train, test = train_test_split(df, test_size=0.2)
train.shape , test.shape

((40339, 3), (10085, 3))

In [12]:
train.to_csv("ecommerce.train", columns=["category_description"], index=False, header=False)
test.to_csv("ecommerce.test", columns=["category_description"], index=False, header=False)

## Train the model and evaluate performance

In [15]:
model = fasttext.train_supervised(input="ecommerce.train")
model.test("ecommerce.test")

(10085, 0.9699553792761527, 0.9699553792761527)

# prediction for few product descriptions

In [18]:
labels, probabilities = model.predict(["wintech assemble desktop pc cpu 500 gb sata hdd 4 gb ram intel c2d processor 3"])
print(labels, probabilities)

[['__label__electronics']] [array([0.990713], dtype=float32)]


In [19]:
model.predict(["ockey men's cotton t shirt fabric details 80 cotton 20 polyester super combed cotton rich fabric"])

([['__label__clothing_accessories']], [array([1.00001], dtype=float32)])

In [23]:
model.get_nearest_neighbors("sony")

[(0.9993138313293457, 'oc'),
 (0.9992992877960205, 'replicator'),
 (0.9992895722389221, 'gofree'),
 (0.9992849230766296, '240gb'),
 (0.9992838501930237, '810g'),
 (0.9992836117744446, 'soundboss'),
 (0.9992832541465759, 'posterguy'),
 (0.9992820024490356, 'airbag'),
 (0.9992810487747192, 'motobike'),
 (0.9992636442184448, 'ipx8')]